In [62]:
# Import required packages
using DifferentialEquations     # For solving differential equations
using LinearAlgebra             # Provides linear algebra functionalities
using SparseArrays              # For efficient storage of sparse matrices
#using KLU                       # Sparse LU factorization solver
using IterativeSolvers          # Iterative algorithms for linear systems
using SparseDiffTools           # Tools for differentiating sparse functions
using Plots                     # For plotting results
using YAML                      # For parsing YAML files

# Ensure all packages are installed. If not, instruct the user to install them.
# Example: Pkg.add("DifferentialEquations"), etc.

include("Chemistry.jl") # Chemistry.jl is the actual program

atom_counter (generic function with 1 method)

In [69]:
# Set up the ODE problem
const FILENAME = "LaurentMechanism.yaml"

# Load the mechanism file
mechanism_data = load_mechanism_data(FILENAME)

# Process the chemical data
species_list, reaction_list, species_index_map = process_chemical_data(mechanism_data)

# Number of species and reactions
const n_species = length(species_list)
const n_vars = n_species + 1
const n_reactions = length(reaction_list)

# Build the stoichiometric matrix and kinetics list
S = build_stoichiometric_matrix(reaction_list, n_species)
kinetics_list = build_kinetics_list(reaction_list)

kinetics_list.elementary_kinetics[4]

ElementaryKinetics(35700.0, 2.4, -2109.94, false, true, Tuple{Int64, Float64}[], [5], [2.0], [2, 6], [1.0, 1.0])

In [66]:
config = ChemistryConfig(
        temperature = 900.0, # Initial temperature in K
        pressure = 1e5,      # Initial pressure in Pa
        fuel_mixture = Dict(
            "CH4" => 0.00, 
            "H2" => 0.99/7
            ),
        air_percentage = 6/7 # 90% Air
)

X0 = initialize_concentrations(config, species_list, species_index_map)

54-element Vector{Float64}:
 900.0
   1.8893583863064694
   0.0
   0.0
   2.398454591489412
   0.0
   0.022901313773411747
   0.0
   0.0
   0.0
   0.0
   0.0
   0.0
   ⋮
   0.0
   0.0
   0.0
   0.0
   0.0
   0.0
   8.941130923415415
   0.10694913532183287
   0.0
   0.0
   0.0
   0.0

In [57]:
function dT(X::Vector{Float64}, r::Vector{Float64}, S::Array{Float64,2}, species_list::Vector{Species})
    n_species = length(species_list)
    n_reactions = length(r)

    h0_vec = zeros(n_reactions)      # Enthalpy change per reaction (J/mol)
    cp_vec = zeros(n_species)        # Heat capacity per species (J/(mol·K))

    T = X[1]                         # Temperature in K
    concentrations = X[2:end]        # Species concentrations in mol/m³

    # Pre-compute species enthalpies and heat capacities
    h_species = zeros(n_species)     # Enthalpy for each species (J/mol)
    for (species_index, species) in enumerate(species_list)
        cp_vec[species_index] = species_cp(T, species.thermo)
        h_species[species_index] = h0(T, species.thermo)
    end

    # Compute h0_vec for reactions
    for reaction_index in 1:n_reactions
        # Sum over species: stoichiometric coefficient * species enthalpy
        for species_index in 1:n_species
            stoich_coeff = S[species_index, reaction_index]
            if stoich_coeff != 0.0
                h0_vec[reaction_index] += stoich_coeff * h_species[species_index]
            end
        end
    end

    # Compute the total enthalpy change rate (J/(m³·s))
    dH = sum(r .* h0_vec)

    # Compute the total heat capacity of the mixture (J/(m³·K))
    c_p = sum(concentrations .* cp_vec)

    # Compute temperature rate of change (K/s)
    dT_dt = -dH / c_p

    return dT_dt
end
    
# Define the ODE function
function reaction_ode!(dX, X, p, t)
    # Unpack parameters
    S, kinetics_tuples, species_list, reaction_list, species_index_map = p

    r = zeros(length(reaction_list))

    # Compute reaction rates
    compute_reaction_rates!(r, X, kinetics_tuples, species_list)

    # Compute temperature rate of change
    dT_dt = dT(X, r, S, species_list)

    # Compute species concentration rate of change
    dC_dt = S * r  # dC/dt = S * r

    # Populate the derivative vector
    dX[1] = dT_dt       # Temperature derivative
    @views dX[2:end] .= dC_dt  # Species concentration derivatives
end

reaction_ode! (generic function with 1 method)

In [67]:
# Define time span and solver settings
tspan = (0.0, 1e-6)
abstol, reltol = 1e-4, 1e-6

# Set up the ODE problem
params = S, kinetics_list, species_list, reaction_list, species_index_map
problem = ODEProblem(reaction_ode!, X0, tspan, params)

# Solve the ODE problem
sol = solve(problem,
            Rosenbrock23(autodiff = AutoFiniteDiff()),
            verbose=false,
            abstol=abstol,
            reltol=reltol)

retcode: Success
Interpolation: specialized 2nd order "free" stiffness-aware interpolation
t: 132-element Vector{Float64}:
 0.0
 2.5917820942270893e-13
 2.8509603036497983e-12
 2.876878124592069e-11
 2.879469906686296e-10
 2.8797290848957185e-9
 2.8181738387869244e-8
 4.8524563583691035e-8
 8.472831325526237e-8
 1.0544184424372966e-7
 1.396840600155404e-7
 1.5906742321234092e-7
 1.8909419705832482e-7
 ⋮
 3.899353719462311e-7
 3.89983947723829e-7
 3.9019008034325444e-7
 3.9039621296267986e-7
 3.90905383892744e-7
 3.9170841219677413e-7
 3.95309273510029e-7
 4.0730131238952954e-7
 4.5473146873599775e-7
 7.721500764118936e-7
 9.921376925718492e-7
 1.0e-6
u: 132-element Vector{Vector{Float64}}:
 [900.0, 1.8893583863064694, 0.0, 0.0, 2.398454591489412, 0.0, 0.022901313773411747, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 8.941130923415415, 0.10694913532183287, 0.0, 0.0, 0.0, 0.0]
 [899.9999999998676, 1.8893583863034196, 6.292162475999224e-17, 8.620570997436658e-17, 2.3984545914833126, 2.485203240

In [75]:
# Extract the solution arrays
t = sol.t
T = sol[1, :]                  # Temperature over time
concentrations = sol[2:end, :]  # Species concentrations over time

# Plot results
plot1 = plot(t, T,
         xlabel = "Time (s)",
         ylabel = "Temperature (K)",
         legend = false)
plot2 = plot(t, concentrations',
         xlabel = "Time (s)",
         ylabel = "Concentration (mol/m³)",
         legend = false)

plot(plot1, plot2, layout = (2,1))
savefig("new_h2_single_node")

"C:\\Users\\jelte\\chemicalcombustion\\new_h2_single_node.png"

In [20]:
h0(298.15, species_list[species_index_map["H2O"]].thermo)

-241824.60364649128